# RF training — Google Colab

Trains four `RandomForestClassifier` models that are too large for local 8GB RAM:
- `fp_only` (2048-dim Morgan fingerprint), random split
- `fp_esm` (Morgan fingerprint + ESM2 embedding, 2688-dim), random / held-out target / scaffold splits

## Before running

Upload the following files from your local repo to a folder named `bindscape_cache` inside **My Drive** on Google Drive:

| Local path | Drive path |
|---|---|
| `cache/fp_smiles.json` | `My Drive/bindscape_cache/fp_smiles.json` |
| `cache/fp_matrix.npy` | `My Drive/bindscape_cache/fp_matrix.npy` |
| `cache/protein_ids.json` | `My Drive/bindscape_cache/protein_ids.json` |
| `cache/protein_embeddings.npy` | `My Drive/bindscape_cache/protein_embeddings.npy` |
| `data/bindingdb_scaffold.pkl` | `My Drive/bindscape_cache/bindingdb_scaffold.pkl` |

After the notebook finishes, download from Drive:
- `My Drive/bindscape_cache/RF_fp_only_random.joblib`
- `My Drive/bindscape_cache/RF_fp_esm_random.joblib`
- `My Drive/bindscape_cache/RF_fp_esm_target.joblib`
- `My Drive/bindscape_cache/RF_fp_esm_scaffold.joblib`

Place them in your local `models/` directory. Re-run the main notebook — the RF cells will load them automatically.

In [ ]:
!pip install -q scikit-learn numpy pandas joblib rdkit psutil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/bindscape_cache'
import os
assert os.path.isdir(DRIVE), f'Folder not found: {DRIVE} — create it in Drive and upload the files listed above'

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SMILES_COL     = 'Ligand SMILES'
UNIPROT_COL    = 'UniProt (SwissProt) Primary ID of Target Chain 1'

DRIVE = Path('/content/drive/MyDrive/bindscape_cache')

# Load fingerprint cache
with open(DRIVE / 'fp_smiles.json') as f:
    fp_smiles = json.load(f)
fp_mat = np.load(DRIVE / 'fp_matrix.npy')
fp_cache = dict(zip(fp_smiles, fp_mat))
print(f'Fingerprints loaded: {len(fp_cache)}')

# Load embedding cache
with open(DRIVE / 'protein_ids.json') as f:
    protein_ids = json.load(f)
emb_mat = np.load(DRIVE / 'protein_embeddings.npy')
emb_cache = dict(zip(protein_ids, emb_mat))
print(f'Embeddings loaded: {len(emb_cache)}')

# Load filtered dataframe (includes scaffold column)
df = pd.read_pickle(DRIVE / 'bindingdb_scaffold.pkl')
print(f'DataFrame loaded: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

assert SMILES_COL in df.columns, f'Missing column: {SMILES_COL}'
assert UNIPROT_COL in df.columns, f'Missing column: {UNIPROT_COL}'
assert 'label' in df.columns
assert 'scaffold' in df.columns

In [ ]:
# Feature matrices — must match main notebook exactly
y = df['label'].values.astype(np.int32)
FP  = np.stack([fp_cache[s]  for s in df[SMILES_COL]])
ESM = np.stack([emb_cache[u] for u in df[UNIPROT_COL]])
print(f'FP: {FP.shape}, ESM: {ESM.shape}, y: {y.shape}')

n_pos = (y == 1).sum()
n_neg = (y == 0).sum()
ratio = n_pos / max(n_neg, 1)
print(f'Positives: {n_pos}, Negatives: {n_neg}, ratio {ratio:.2f}:1')

In [ ]:
# Splits — identical logic and seeds as main notebook
N = len(df)
idx_all = np.arange(N)

# Split 1 — random
idx_trainval, idx_test1 = train_test_split(idx_all, test_size=0.10, random_state=42, stratify=y)
idx_train1, idx_val1   = train_test_split(idx_trainval, test_size=0.10/0.90, random_state=42, stratify=y[idx_trainval])

# Split 2 — held-out target
unique_targets = df[UNIPROT_COL].unique()
np.random.seed(42)
np.random.shuffle(unique_targets)
n_test_targets = max(1, int(0.20 * len(unique_targets)))
test_targets  = set(unique_targets[:n_test_targets])
train_targets = set(unique_targets[n_test_targets:])
idx_train2 = np.where(df[UNIPROT_COL].isin(train_targets).values)[0]
idx_test2  = np.where(df[UNIPROT_COL].isin(test_targets).values)[0]

# Split 3 — scaffold
unique_scaffolds = df['scaffold'].unique()
np.random.seed(42)
np.random.shuffle(unique_scaffolds)
n_test_scaffolds = max(1, int(0.20 * len(unique_scaffolds)))
test_scaffolds = set(unique_scaffolds[:n_test_scaffolds])
idx_train3 = np.where(~df['scaffold'].isin(test_scaffolds).values)[0]
idx_test3  = np.where(df['scaffold'].isin(test_scaffolds).values)[0]

print(f'Split 1 — train:{len(idx_train1)} val:{len(idx_val1)} test:{len(idx_test1)}')
print(f'Split 2 — train:{len(idx_train2)} test:{len(idx_test2)}')
print(f'Split 3 — train:{len(idx_train3)} test:{len(idx_test3)}')

In [ ]:
# RF fp_only — random split (FP is binary, no standardization needed)
print('Fitting RF fp_only — random split...')
rf_fp_only = RandomForestClassifier(
    n_estimators=300, max_features='sqrt', random_state=42,
    n_jobs=-1, class_weight='balanced_subsample'
).fit(FP[idx_train1], y[idx_train1])
out_path = DRIVE / 'RF_fp_only_random.joblib'
joblib.dump(rf_fp_only, out_path)
print(f'Saved: {out_path}')

In [ ]:
# Standardize ESM using scaler fitted on random split train set
# This must match main notebook: scaler_esm = StandardScaler().fit(ESM[idx_train1])
scaler_esm = StandardScaler().fit(ESM[idx_train1])
ESM_s = scaler_esm.transform(ESM)

X_fp_esm = np.hstack([FP, ESM_s])
print(f'fp_esm feature matrix: {X_fp_esm.shape}')

In [ ]:
import psutil
avail_gb = psutil.virtual_memory().available / 1e9
print(f'Available RAM: {avail_gb:.1f} GB')

def fit_rf(X_train, y_train):
    return RandomForestClassifier(
        n_estimators=300, max_features='sqrt', random_state=42,
        n_jobs=-1, class_weight='balanced_subsample'
    ).fit(X_train, y_train)

# Random split
print('Fitting RF fp_esm — random split...')
rf_rand = fit_rf(X_fp_esm[idx_train1], y[idx_train1])
out_path = DRIVE / 'RF_fp_esm_random.joblib'
joblib.dump(rf_rand, out_path)
print(f'Saved: {out_path}')

In [ ]:
# Target split
print('Fitting RF fp_esm — held-out target split...')
rf_targ = fit_rf(X_fp_esm[idx_train2], y[idx_train2])
out_path = DRIVE / 'RF_fp_esm_target.joblib'
joblib.dump(rf_targ, out_path)
print(f'Saved: {out_path}')

In [ ]:
# Scaffold split
print('Fitting RF fp_esm — scaffold split...')
rf_scaf = fit_rf(X_fp_esm[idx_train3], y[idx_train3])
out_path = DRIVE / 'RF_fp_esm_scaffold.joblib'
joblib.dump(rf_scaf, out_path)
print(f'Saved: {out_path}')

## Done

Download from `My Drive/bindscape_cache/`:
- `RF_fp_only_random.joblib`
- `RF_fp_esm_random.joblib`
- `RF_fp_esm_target.joblib`
- `RF_fp_esm_scaffold.joblib`

Place all four in your local `models/` directory and re-run the main `notebook.ipynb` from section 7 onward.